# Notebook ١٠: Post-training (Instruction Tuning وLoRA وDPO)

تدريب بعد الـ pretraining على GPT-2 حقيقي وKeras. كل خلية اتشغلت فعلًا.

## ١. Instruction tuning: الـ loss على الرد بس

نفس تدريب الـ language model، بس بنحسب الخسارة على كلمات الرد، والتعليمة بتبقى سياق.

In [ ]:
# نخفي التحذيرات
import warnings; warnings.filterwarnings("ignore")
# backend موديلات Hugging Face
import torch
# الـ tokenizer والموديل
from transformers import AutoTokenizer, AutoModelForCausalLM
# tokenizer GPT-2
tok = AutoTokenizer.from_pretrained("gpt2")
# موديل GPT-2
lm = AutoModelForCausalLM.from_pretrained("gpt2").eval()
# التعليمة
prompt = "Instruction: Who is the top scorer? Answer:"
# الرد المطلوب
answer = " Haaland"
# أرقام التعليمة
p_ids = tok(prompt)["input_ids"]
# أرقام الرد
a_ids = tok(answer)["input_ids"]
# التعليمة والرد ورا بعض
ids = torch.tensor([p_ids + a_ids])
# -100 يعني تجاهل الخسارة هنا
labels = torch.tensor([[-100] * len(p_ids) + a_ids])
# أعداد التوكنز
print(len(p_ids), "prompt tokens +", len(a_ids), "answer tokens")
# الـ labels: التعليمة متجاهلة
print(labels[0].tolist())


In [ ]:
# من غير gradients
with torch.no_grad():
# خسارة الرد بس
    loss_ans = lm(input_ids=ids, labels=labels).loss
# خسارة كل التوكنز
    loss_all = lm(input_ids=ids, labels=ids).loss
# خسارة الرد
print("loss on answer only:", round(float(loss_ans), 2))
# خسارة الكل
print("loss on all tokens: ", round(float(loss_all), 2))


## ٢. LoRA بـ Keras

نجمّد W ونتدرب على مصفوفتين صغيرتين A وB: h = xW + xAB.

In [ ]:
# المصفوفات وTensorFlow
import numpy as np, tensorflow as tf
# Keras
from tensorflow import keras
# seed ثابت
keras.utils.set_random_seed(0)
# طبقة Dense بـ LoRA
class LoRADense(keras.layers.Layer):
# الوحدات والـ rank
    def __init__(self, units, rank=4, **kw):
# تهيئة الطبقة الأب
        super().__init__(**kw)
# نخزن القيم
        self.units, self.rank = units, rank
# بتبني الأوزان
    def build(self, shape):
# بعد المدخل
        d = shape[-1]
# W متجمدة
        self.W = self.add_weight(shape=(d, self.units), initializer="glorot_uniform", trainable=False, name="W")
# A قابلة للتدريب
        self.A = self.add_weight(shape=(d, self.rank), initializer="glorot_uniform", name="A")
# B بصفر في البداية
        self.B = self.add_weight(shape=(self.rank, self.units), initializer="zeros", name="B")
# الحساب
    def call(self, x):
# xW + xAB
        return x @ self.W + (x @ self.A) @ self.B
# طبقة 256 وحدة
layer = LoRADense(256, rank=4)
# نبنيها بمدخل 256
layer.build((None, 256))
# أوزان قابلة للتدريب
tr = sum(int(np.prod(w.shape)) for w in layer.trainable_weights)
# الإجمالي
tot = tr + int(np.prod(layer.W.shape))
# النسبة
print("trainable:", tr, "of", tot, f"({tr / tot * 100:.1f}%)")


In [ ]:
# مدخلات تجريبية
x = tf.random.normal((4, 256))
# B=0 فالخرج في البداية = خرج W بس
print(np.allclose(layer(x).numpy(), (x @ layer.W).numpy()))
# هدف: تغيير low-rank (فرضية LoRA)
W_true = tf.random.normal((256, 4)) @ tf.random.normal((4, 256)) * 0.05
# داتا تدريب
X = tf.random.normal((256, 256))
# الخرج المطلوب
Y = X @ (layer.W + W_true)
# المحسّن
opt = keras.optimizers.Adam(0.01)
# نحفظ W قبل التدريب
w_before = layer.W.numpy().copy()
# 201 خطوة
for step in range(201):
# نسجل الحسابات
    with tf.GradientTape() as g:
# MSE
        loss = tf.reduce_mean((layer(X) - Y) ** 2)
# gradients لـ A وB بس
    gr = g.gradient(loss, layer.trainable_weights)
# نحدّث A وB
    opt.apply_gradients(zip(gr, layer.trainable_weights))
# كل 100 خطوة
    if step % 100 == 0:
# الخسارة
        print(step, round(float(loss), 4))
# W ما اتغيرتش
print("W unchanged:", np.array_equal(w_before, layer.W.numpy()))


## ٣. Preferences وDPO

بنعلّم الموديل يفضّل الرد الأحسن من غير reward model منفصل.

In [ ]:
# log احتمال الرد بعد التعليمة
def seq_logprob(model, prompt, resp):
# نحول لأرقام
    p = tok(prompt)["input_ids"]; r = tok(resp)["input_ids"]
# الكل ورا بعض
    ids = torch.tensor([p + r])
# log احتمالات
    lp = torch.log_softmax(model(input_ids=ids).logits[0, :-1], -1)
# التوكنز المتوقعة
    tgt = ids[0, 1:]
# مجموع log احتمال توكنز الرد
    return lp[range(len(tgt)), tgt][len(p) - 1:].sum()
# السؤال
q = "Question: Is Salah injured? Answer:"
# الرد المفضل
good = " I am not sure, please check the official club news."
# الرد المرفوض (تأكيد من غير مصدر)
bad = " Yes he is definitely out for the season."
# من غير gradients
with torch.no_grad():
# log احتمال كل رد
    print(round(float(seq_logprob(lm, q, good)), 2), round(float(seq_logprob(lm, q, bad)), 2))


In [ ]:
# المرجع من غير gradients
with torch.no_grad():
# log احتمالات الموديل قبل التدريب (المرجع)
    rg, rb = seq_logprob(lm, q, good), seq_logprob(lm, q, bad)
# فجوة الجيد عن السيئ قبل التدريب
b = float(rg - rb)
# هنحدّث نفس الموديل من غير dropout (القيم المرجعية اتحفظت فوق)
policy = lm.eval()
# نجمّد كل الأوزان
for p in policy.parameters():
# بدون gradient
    p.requires_grad_(False)
# آخر بلوكين بس
train_params = [p for blk in policy.transformer.h[-2:] for p in blk.parameters()]
# نفك تجميدهم
for p in train_params:
# يتدربوا
    p.requires_grad_(True)
# SGD على آخر بلوكين (عشان الذاكرة محدودة)
opt = torch.optim.SGD(train_params, lr=1e-3)
# قوة عقوبة الابتعاد عن المرجع
beta = 0.1
# 6 خطوات
for step in range(6):
# log احتمالات الموديل
    pg, pb = seq_logprob(policy, q, good), seq_logprob(policy, q, bad)
# الفرق بين تفضيل الجيد والسيئ
    margin = beta * ((pg - rg) - (pb - rb))
# DPO loss
    loss = -torch.nn.functional.logsigmoid(margin)
# تحديث الأوزان
    opt.zero_grad(); loss.backward(); opt.step()
# نطبع الخسارة والفرق
    print(step, "loss:", round(float(loss), 4), "margin:", round(float(margin), 4))


In [ ]:
# من غير gradients
with torch.no_grad():
# فرق log احتمال بعد التدريب
    a = float(seq_logprob(policy.eval(), q, good) - seq_logprob(policy, q, bad))
# الفجوة قبل وبعد
print("good-bad log-prob gap before:", round(b, 2), "after:", round(a, 2))


الملاحظة: الـ DPO loss بدأ 0.6931 (= log 2) لأن الموديل والمرجع متطابقين وقتها، وبعدين نزل. الفجوة بين الرد الجيد والسيئ اتحسنت من -16.96 لـ -16.12، تحسن صغير لأن دي 6 خطوات على زوج واحد وعلى آخر بلوكين بس. ده شرح للميكانيزم، مش تدريب حقيقي.